# Walmart Inc. (WMT.O) — Valoración de derivados y estructura convertible

**Proyecto Final · Productos Derivados y Administración de Riesgos Corporativos**
EGADE Business School · Maestría en Finanzas · Dr. Raymundo Díaz Robles

**Equipo 6 · Grupo 40**
Eduardo Ayala Saldaña (A01192714) · Mariann Antonelli Morales Tobias (A01452078) · Ulises Salinas López (A00814787)

Datos: LSEG Workspace, corte al 1 de noviembre de 2026.
Este notebook no requiere instalar nada: usa solo la biblioteca estándar de Python.

In [1]:
import math
from dataclasses import dataclass

# Parámetros de mercado (LSEG Workspace, corte 1-nov-2026)
S0       = 123.51      # Cierre de WMT.O del 30-oct-2026
DIV      = 0.25        # Dividendo trimestral declarado
T_DIV    = 40/365      # Días al ex-dividendo (11-dic-2026)
T_EXP    = 47/365      # Días al vencimiento (18-dic-2026)
RF       = 0.0403      # Tasa libre de riesgo a 47 días, curva del Tesoro
SIGMA_LP = 0.29        # Volatilidad implícita de LEAPs (ene-2029)
Q_DIV    = 0.0081      # Dividend yield prospectivo

print(f"Spot            {S0}")
print(f"Plazo           {T_EXP*365:.0f} días  ({T_EXP:.5f} años)")
print(f"Ex-dividendo    {T_DIV*365:.0f} días")
print(f"Tasa libre      {RF:.2%}")

Spot            123.51
Plazo           47 días  (0.12877 años)
Ex-dividendo    40 días
Tasa libre      4.03%


## 1. Black-Scholes-Merton con dividendo discreto

El dividendo se descuenta del spot antes de valuar (modelo *escrow*, Hull capítulo 15).

In [2]:
def _cdf(x):  return 0.5*(1.0+math.erf(x/math.sqrt(2.0)))
def _pdf(x):  return math.exp(-0.5*x*x)/math.sqrt(2.0*math.pi)

def vp_div(divs, r):
    return sum(d*math.exp(-r*t) for d, t in divs)

def black_scholes(S, K, T, r, sigma, tipo="call", divs=()):
    Sa = S - vp_div(divs, r)
    if T <= 0 or sigma <= 0:
        return max(Sa-K, 0.0) if tipo == "call" else max(K-Sa, 0.0)
    d1 = (math.log(Sa/K) + (r + 0.5*sigma**2)*T)/(sigma*math.sqrt(T))
    d2 = d1 - sigma*math.sqrt(T)
    if tipo == "call":
        return Sa*_cdf(d1) - K*math.exp(-r*T)*_cdf(d2)
    return K*math.exp(-r*T)*_cdf(-d2) - Sa*_cdf(-d1)

def vol_implicita(precio, S, K, T, r, tipo="call", divs=(), lo=1e-4, hi=5.0):
    f = lambda s: black_scholes(S, K, T, r, s, tipo, divs) - precio
    if f(lo)*f(hi) > 0: return None
    for _ in range(200):
        mid = 0.5*(lo+hi)
        if f(lo)*f(mid) <= 0: hi = mid
        else:                 lo = mid
    return 0.5*(lo+hi)

DIVS = ((DIV, T_DIV),)
print("Call K=125 :", round(black_scholes(S0, 125, T_EXP, RF, 0.318928, "call", DIVS), 5))
print("Put  K=125 :", round(black_scholes(S0, 125, T_EXP, RF, 0.323758, "put",  DIVS), 5))

Call K=125 : 5.12034
Put  K=125 : 6.29747


## 2. Árbol binomial Cox-Ross-Rubinstein

Dividendo discreto por el método de *escrow*: el árbol se construye sobre `S* = S − VP(dividendos)`
y el dividendo se reincorpora en los nodos anteriores a la fecha ex-dividendo, de modo que el valor
intrínseco de cada nodo refleja el precio observable de la acción.

In [3]:
def arbol_crr(S, K, T, r, sigma, N, tipo="call", americana=True, divs=()):
    dt = T/N
    u  = math.exp(sigma*math.sqrt(dt)); d = 1.0/u
    p  = (math.exp(r*dt) - d)/(u - d);  disc = math.exp(-r*dt)
    Sx = S - vp_div(divs, r)

    def escrow(i):
        t = i*dt
        return sum(dv*math.exp(-r*(td-t)) for dv, td in divs if td > t)

    intr = (lambda s: max(s-K, 0.0)) if tipo == "call" else (lambda s: max(K-s, 0.0))
    V = [intr(Sx*(u**j)*(d**(N-j)) + escrow(N)) for j in range(N+1)]
    for i in range(N-1, -1, -1):
        e = escrow(i); Vn = []
        for j in range(i+1):
            cont = disc*(p*V[j+1] + (1-p)*V[j])
            Vn.append(max(cont, intr(Sx*(u**j)*(d**(i-j)) + e)) if americana else cont)
        V = Vn
    return V[0]

def valuar(S, K, T, r, sigma, N, tipo="call", divs=()):
    eu = arbol_crr(S, K, T, r, sigma, N, tipo, False, divs)
    am = arbol_crr(S, K, T, r, sigma, N, tipo, True,  divs)
    return eu, am, am - eu

# Validación contra el ejemplo de referencia de Hull (capítulo 21)
# Put americana: S=50, K=50, r=10%, sigma=40%, T=5 meses, N=5  ->  4.4886
print("Put americana de Hull, N=5 :", round(arbol_crr(50, 50, 5/12, 0.10, 0.40, 5, "put"), 4))
print("Referencia del libro       : 4.4886")

Put americana de Hull, N=5 : 4.4885
Referencia del libro       : 4.4886


## 3. Convergencia del árbol a Black-Scholes

In [4]:
bs = black_scholes(S0, 125, T_EXP, RF, 0.318928, "call", DIVS)
print(f"Black-Scholes = {bs:.6f}\n")
print(f"{'N':>6} {'Árbol':>12} {'Diferencia':>12}")
for N in (10, 50, 100, 500, 1000, 2000):
    v = arbol_crr(S0, 125, T_EXP, RF, 0.318928, N, "call", False, DIVS)
    print(f"{N:>6} {v:>12.6f} {v-bs:>+12.6f}")

Black-Scholes = 5.120340

     N        Árbol   Diferencia
    10     5.152286    +0.031946
    50     5.147135    +0.026796
   100     5.132872    +0.012532
   500     5.122728    +0.002389
  1000     5.119597    -0.000743
  2000     5.120719    +0.000379


## 4. El dividendo como única fuente de ejercicio anticipado

Experimento controlado sobre la opción de compra con precio de ejercicio 125.
Todo permanece constante salvo el dividendo.

In [5]:
print(f"{'Dividendo':>10} {'Europea':>10} {'Americana':>11} {'Premio':>11}")
for d in (0.00, 0.25, 0.75, 1.50, 2.50, 4.00):
    dv = ((d, T_DIV),) if d > 0 else ()
    eu, am, pr = valuar(S0, 125, T_EXP, RF, 0.318928, 800, "call", dv)
    print(f"${d:>9.2f} {eu:>10.5f} {am:>11.5f} {pr:>11.6f}")
print()
print("Con dividendo cero el premio es exactamente cero: la call americana vale")
print("lo mismo que la europea (Merton, 1973). Con el dividendo real de Walmart")
print("el premio aparece y crece de forma monótona con su magnitud.")

 Dividendo    Europea   Americana      Premio
$     0.00    5.24543     5.24543    0.000000
$     0.25    5.12133     5.14785    0.026521
$     0.75    4.87959     5.03494    0.155342
$     1.50    4.52832     4.91236    0.384045
$     2.50    4.08745     4.79488    0.707437
$     4.00    3.48222     4.66727    1.185057

Con dividendo cero el premio es exactamente cero: la call americana vale
lo mismo que la europea (Merton, 1973). Con el dividendo real de Walmart
el premio aparece y crece de forma monótona con su magnitud.


## 5. Curva del Tesoro y valoración del convertible

Curva de LSEG/TRADEWEB al 1 de noviembre de 2026, trece plazos.

In [6]:
CURVA = [(0.0833,3.9715),(0.1667,4.0835),(0.25,4.0820),(0.3333,4.1905),
         (0.5,4.2690),(1,4.3755),(2,4.6030),(3,4.6710),(5,4.7320),
         (7,4.8230),(10,4.9350),(20,5.3420),(30,5.3145)]

def ust(T):
    if T <= CURVA[0][0]:  return CURVA[0][1]/100
    if T >= CURVA[-1][0]: return CURVA[-1][1]/100
    for (t0,y0),(t1,y1) in zip(CURVA, CURVA[1:]):
        if t0 <= T <= t1:
            return (y0 + (y1-y0)*(T-t0)/(t1-t0))/100

def bono_recto(cupon, anios, ytm, face=100.0, freq=2):
    n = int(round(anios*freq)); c = face*cupon/freq; y = ytm/freq
    return sum(c/(1+y)**k for k in range(1, n+1)) + face/(1+y)**n

def bs_q(S, K, T, r, sigma, q):
    d1 = (math.log(S/K) + (r - q + 0.5*sigma**2)*T)/(sigma*math.sqrt(T))
    d2 = d1 - sigma*math.sqrt(T)
    return S*math.exp(-q*T)*_cdf(d1) - K*math.exp(-r*T)*_cdf(d2)

def convertible(S, anios, ytm, sigma, q, prima, cupon, face=100.0):
    K = S*(1+prima); ratio = face/K
    return dict(K=K, ratio=ratio, bono=bono_recto(cupon, anios, ytm, face),
                opcion=ratio*bs_q(S, K, anios, ust(anios), sigma, q))

def cupon_a_la_par(S, anios, ytm, sigma, q, prima, face=100.0):
    lo, hi = 0.0, ytm*2
    for _ in range(200):
        mid = (lo+hi)/2
        v = convertible(S, anios, ytm, sigma, q, prima, mid, face)
        if v["bono"]+v["opcion"] > face: hi = mid
        else:                            lo = mid
    return (lo+hi)/2

print(f"Tasa a 47 días : {ust(47/365):.4%}")
print(f"Tasa a 6 años  : {ust(6):.4%}")

Tasa a 47 días : 4.0326%
Tasa a 6 años  : 4.7775%


In [7]:
CV_T, CV_YTM, CV_PRIMA, CV_EMISION = 6, 0.0514, 0.25, 1500.0

cp = cupon_a_la_par(S0, CV_T, CV_YTM, SIGMA_LP, Q_DIV, CV_PRIMA)
cv = convertible(S0, CV_T, CV_YTM, SIGMA_LP, Q_DIV, CV_PRIMA, cp)

print(f"Precio de conversión        ${cv['K']:.2f}")
print(f"Acciones por 100 de nominal  {cv['ratio']:.4f}")
print(f"Cupón a la par               {cp:.3%}   (deuda recta: {CV_YTM:.3%})")
print(f"Componente bono recto        {cv['bono']:.3f}")
print(f"Componente opción            {cv['opcion']:.3f}")
print(f"Valor total                  {cv['bono']+cv['opcion']:.3f}")

Precio de conversión        $154.39
Acciones por 100 de nominal  0.6477
Cupón a la par               0.918%   (deuda recta: 5.140%)
Componente bono recto        78.438
Componente opción            21.562
Valor total                  100.000


## 6. La demostración de que no hay *free lunch*

El enunciado del proyecto pide demostrar que *"toda mejora del cupón o de la prima de conversión
tiene un precio en derivados"*. Aquí está el cálculo.

In [8]:
af  = sum(1/(1+CV_YTM/2)**k for k in range(1, CV_T*2+1))
pv  = (CV_YTM - cp)*CV_EMISION/2*af
opc = cv["opcion"]/100*CV_EMISION

print(f"Valor presente del ahorro de cupón   ${pv:>8,.1f} M")
print(f"Valor de la opción entregada         ${opc:>8,.1f} M")
print(f"Diferencia                           ${pv-opc:>8,.1f} M")
print()
print("Son el mismo número. El convertible cotiza a la par precisamente porque")
print("el descuento de cupón equivale, peso por peso, a la opción entregada.")
print()
print(f"{'Prima':>7} {'P. conversión':>14} {'Cupón':>9} {'Opción':>9}")
for p in (0.15, 0.20, 0.25, 0.30, 0.40, 0.50):
    c = cupon_a_la_par(S0, CV_T, CV_YTM, SIGMA_LP, Q_DIV, p)
    v = convertible(S0, CV_T, CV_YTM, SIGMA_LP, Q_DIV, p, c)
    print(f"{p:>6.0%} {'$'+format(v['K'],'.2f'):>14} {c:>9.3%} {v['opcion']:>9.2f}")

Valor presente del ahorro de cupón   $   323.4 M
Valor de la opción entregada         $   323.4 M
Diferencia                           $    -0.0 M

Son el mismo número. El convertible cotiza a la par precisamente porque
el descuento de cupón equivale, peso por peso, a la opción entregada.

  Prima  P. conversión     Cupón    Opción
   15%        $142.04    0.051%     25.99
   20%        $148.21    0.510%     23.65
   25%        $154.39    0.918%     21.56
   30%        $160.56    1.283%     19.70
   40%        $172.91    1.905%     16.52
   50%        $185.27    2.408%     13.95


## 7. Call spread y uso de fondos

In [9]:
acc = CV_EMISION*1e6/(S0*(1+CV_PRIMA))/1e6
K1, K2 = S0*(1+CV_PRIMA), S0*1.60
c1 = bs_q(S0, K1, CV_T, ust(CV_T), SIGMA_LP, Q_DIV)
c2 = bs_q(S0, K2, CV_T, ust(CV_T), SIGMA_LP, Q_DIV)

print(f"Strike comprado (prima 25%)  ${K1:.2f}")
print(f"Strike vendido  (prima 60%)  ${K2:.2f}")
print(f"Acciones subyacentes          {acc:.2f} M")
print(f"Prima pagada                 ${c1*acc:>8,.1f} M")
print(f"Prima cobrada                ${c2*acc:>8,.1f} M")
print(f"COSTO NETO                   ${(c1-c2)*acc:>8,.1f} M   "
      f"({(c1-c2)*acc/CV_EMISION:.2%} de la emisión)")
print()
PX_2051, FACIAL = 57.858, 1500.0
print("Uso de fondos — nota 2.65% con vencimiento 2051")
print(f"  Precio de mercado          {PX_2051:.3f}")
print(f"  Valor facial a retirar     ${FACIAL:>8,.1f} M")
print(f"  Costo de la recompra       ${FACIAL*PX_2051/100:>8,.1f} M")
print(f"  Descuento capturado        ${FACIAL*(1-PX_2051/100):>8,.1f} M")
print(f"  Facial retirado por dólar   {100/PX_2051:.3f}")

Strike comprado (prima 25%)  $154.39
Strike vendido  (prima 60%)  $197.62
Acciones subyacentes          9.72 M
Prima pagada                 $   323.4 M
Prima cobrada                $   227.5 M
COSTO NETO                   $    95.9 M   (6.39% de la emisión)

Uso de fondos — nota 2.65% con vencimiento 2051
  Precio de mercado          57.858
  Valor facial a retirar     $ 1,500.0 M
  Costo de la recompra       $   867.9 M
  Descuento capturado        $   632.1 M
  Facial retirado por dólar   1.728


## Conclusiones

1. **El convertible no genera ahorro económico.** El valor presente del descuento de cupón (323.4 M)
   iguala exactamente el valor de la opción entregada al tenedor (323.4 M). El emisor vende
   optionalidad a precio de mercado y la cobra en forma de cupón reducido.

2. **El dividendo es la única fuente de ejercicio anticipado en una opción de compra americana.**
   Con dividendo cero el premio es exactamente cero, lo que verifica el resultado de Merton (1973)
   con datos de mercado.

3. **Lo que sí tiene valor es la recompra a descuento.** Retirar 1,500 M de valor facial de la nota
   2.65% de 2051 cuesta 867.9 M y captura un descuento de 632.1 M. Cada dólar retira 1.73 dólares
   de valor facial.

4. **La colocación en dólares canadienses exige un cross-currency swap.** La ventaja de fondeo de
   129 puntos base es de curva, no de crédito, y solo se materializa si la exposición cambiaria
   se neutraliza.